In [10]:
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import autoslo.utils.paths as pu
from autoslo.blueprint_selection.query_timeline import QueryTimeline
from autoslo.models.iconq_model import IconqModel
from autoslo.workload_execution.trace import Trace
from matplotlib.lines import Line2D

from autoslo.featurization.iconq_query_featurizer import IconqQueryFeaturizer


In [21]:
class IsolatedComparer:

    def __init__(self, iconq_query_featurizer_id: str):
        self._iconq_query_featurizer = IconqQueryFeaturizer.load(
            iconq_query_featurizer_id
        )
        self._latency_dict = self._init_real()

    def _init_real(self):
        overall_dict = {}

        for rpu in [4, 8, 16, 32, 64, 128, 256]:
            run_ids = pu.RunLocator.get_run_ids(
                workload_name="benchmarking_workload_99_3_3_shuffled_42",
                blueprint_name=f"single_{rpu}",
            )

            rpu_dict = defaultdict(list)

            for run_id in run_ids:
                trace = Trace(run_id)

                # Assert that all the queries are non-overlapping
                start_times_s = trace.arrival_times()
                end_times_s = trace.completion_times()
                most_recent_end_time = pd.Timestamp(0)
                error_msg = (
                    "Query {query_id} starts at {start_time} which is before the most "
                    "recent end time {most_recent_end_time}."
                )
                for query_id in start_times_s.index:
                    start_time = start_times_s[query_id]
                    end_time = end_times_s[query_id]

                    assert start_time >= most_recent_end_time, error_msg.format(
                        query_id=query_id,
                        start_time=start_time,
                        most_recent_end_time=most_recent_end_time,
                    )
                    most_recent_end_time = end_time

                # Collect latencies per (template, query index) and assert each appears 3
                # times
                latencies_s = trace.latencies_s
                tpcds_temp_and_q_idx = trace.tpcds_temp_and_q_idxs
                latencies_per_temp_and_q = defaultdict(list)
                prev_temp_and_q_idx = None
                for query_id in latencies_s.index:
                    latency = latencies_s[query_id]
                    temp_and_q_idx = tpcds_temp_and_q_idx[query_id]

                    latencies_per_temp_and_q[temp_and_q_idx].append(
                        {
                            "query_id": query_id,
                            "latency_s": latency,
                            "table_cosine_similarity": (
                                0
                                if prev_temp_and_q_idx is None
                                else self._iconq_query_featurizer.table_access_pattern_cosine_similarity_from_tpcds_temp_and_q_idxs(
                                    prev_temp_and_q_idx, temp_and_q_idx, binarize=True
                                )
                            ),
                            "table_coverage": (
                                0
                                if prev_temp_and_q_idx is None
                                else self._iconq_query_featurizer.table_access_pattern_coverage_from_tpcds_temp_and_q_idxs(
                                    prev_temp_and_q_idx, temp_and_q_idx
                                )
                            ),
                        }
                    )
                    prev_temp_and_q_idx = temp_and_q_idx
                for item, data in latencies_per_temp_and_q.items():
                    assert (
                        len(data) == 3
                    ), f"Item {item} appears {len(data)} times instead of 3."

                # Add the latencies to the rpu dict
                for item, data in latencies_per_temp_and_q.items():
                    rpu_dict[item].extend(data)

            overall_dict[rpu] = rpu_dict

        return overall_dict


comparer = IsolatedComparer(iconq_query_featurizer_id="1767994197")

In [18]:
# Create a grid of plots where there are 4 subplots per row. Each subplot corresponds to a
# value of the RPU. Within each subplot, have one datapoint per (template, query index) pair,
# where the x value is its mean latency (log scale) and the y value is its coefficient of variation.
# Use plotly, so that on hover we can see the (template, query index) pair. Save the plot as an html file.

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rpus = sorted(comparer._latency_dict.keys())
num_rows = (len(rpus) + 3) // 4
fig = make_subplots(rows=num_rows, cols=4, subplot_titles=[f"RPU={rpu}" for rpu in rpus])

for i, rpu in enumerate(rpus):
    row = (i // 4) + 1
    col = (i % 4) + 1
    temp_and_q_idxs = []
    median_latencies = []
    covs = []

    for temp_and_q_idx, latencies in comparer._latency_dict[rpu].items():
        latencies = [entry["latency_s"] for entry in latencies]
        mean_latency = sum(latencies) / len(latencies)
        median_latency = sorted(latencies)[len(latencies) // 2]
        variance = sum((x - mean_latency) ** 2 for x in latencies) / len(latencies)
        stddev = variance ** 0.5
        cov = stddev / mean_latency if mean_latency != 0 else 0

        temp_and_q_idxs.append(temp_and_q_idx)
        median_latencies.append(median_latency)
        covs.append(cov)

    scatter = go.Scatter(
        x=median_latencies,
        y=covs,
        mode='markers',
        text=[f"{temp_and_q_idx}" for temp_and_q_idx in temp_and_q_idxs],
        marker=dict(size=10, color='blue', opacity=0.6),
        name=f"RPU={rpu}"
    )

    fig.add_trace(scatter, row=row, col=col)

fig.update_xaxes(title_text="Median Latency (s)", type="log")
fig.update_yaxes(title_text="Coefficient of Variation")

fig.update_layout(height=400 * num_rows, width=1200, title_text="Isolated Query Latency Variability by RPU")
fig.write_html("isolated_query_latency_variability.html")

In [30]:
# Again have a grid where each subplot corresponds to an RPU value. This time, the x axis is the
# deviation of a given query's latency from the mean latency for that (template, query index) pair across
# the 3 runs. The y axis is the table access pattern cosine similarity from the previous query.
# Save the plot as an html file.

# Color the queries based on whether their latency is above or below 10 seconds.
# Make the red and blue dots individually toggleable with tick boxes.

fig = make_subplots(rows=num_rows, cols=4, subplot_titles=[f"RPU={rpu}" for rpu in rpus])
threshold_latency_s = 2.0

for i, rpu in enumerate(rpus):
    row = (i // 4) + 1
    col = (i % 4) + 1
    deviations = []
    coverage_values  = []
    colors = []

    for temp_and_q_idx, data in comparer._latency_dict[rpu].items():
        latencies = [item["latency_s"] for item in data]
        median_latency = sorted(latencies)[len(latencies) // 2]

        for item in data:
            deviation = item["latency_s"] - median_latency
            deviations.append(deviation)
            coverage_values.append(item["table_coverage"])
            colors.append('red' if item["latency_s"] > threshold_latency_s else 'blue')



    # Plot the deviations vs coverage values by color so that they can be toggled
    is_red = [c == 'red' for c in colors]
    scatter_red = go.Scatter(
        x=[dev for dev, red in zip(deviations, is_red) if red],
        y=[cov for cov, red in zip(coverage_values, is_red) if red],
        mode='markers',
        marker=dict(size=10, color='red', opacity=0.6),
        name=f"RPU={rpu} Latency > {threshold_latency_s}s"
    )
    scatter_blue = go.Scatter(
        x=[dev for dev, red in zip(deviations, is_red) if not red],
        y=[cov for cov, red in zip(coverage_values, is_red) if not red],
        mode='markers',
        marker=dict(size=10, color='blue', opacity=0.6),
        name=f"RPU={rpu} Latency <= {threshold_latency_s}s"
    )
    fig.add_trace(scatter_red, row=row, col=col)
    fig.add_trace(scatter_blue, row=row, col=col)

    # Also add red and blue linear regression trend lines
    import numpy as np
    from sklearn.linear_model import LinearRegression

    for color, color_name in [('red', f'Latency > {threshold_latency_s}s'), ('blue', f'Latency <= {threshold_latency_s}s')]:
        x_vals = np.array([dev for dev, c in zip(deviations, colors) if c == color]).reshape(-1, 1)
        y_vals = np.array([cov for cov, c in zip(coverage_values, colors) if c == color])

        if len(x_vals) > 1:
            model = LinearRegression()
            model.fit(x_vals, y_vals)
            x_range = np.linspace(min(x_vals), max(x_vals), 100).reshape(-1, 1)
            y_range = model.predict(x_range)

            trend_line = go.Scatter(
                x=x_range.flatten(),
                y=y_range,
                mode='lines',
                line=dict(color=color, dash='dash'),
                name=f"RPU={rpu} Trend {color_name}"
            )
            fig.add_trace(trend_line, row=row, col=col)

# Make the red and blue traces toggleable with tick boxes. One for all the red traces and one for all the blue traces.
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            buttons=[
                dict(
                    label="Show All",
                    method="update",
                    args=[{"visible": [True] * len(fig.data)}],
                ),
                dict(
                    label="Hide Red",
                    method="update",
                    args=[{"visible": [trace.name.endswith(f"Latency <= {threshold_latency_s}s") for trace in fig.data]}],
                ),
                dict(
                    label="Hide Blue",
                    method="update",
                    args=[{"visible": [trace.name.endswith(f"Latency > {threshold_latency_s}s") for trace in fig.data]}],
                ),
            ],
            showactive=True,
            x=1.1,
            y=1.15,
        )
    ]
)


fig.update_xaxes(title_text="Latency Deviation<br>from Median Latency (s)")
fig.update_yaxes(title_text="Table Access Pattern Coverage")
fig.update_layout(height=400 * num_rows, width=1200, title_text="Isolated Query Latency Deviation vs Table Access Pattern Coverage by RPU")
fig.write_html("isolated_query_latency_deviation_vs_table_access_pattern_coverage.html")